# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [ ]:
%load_ext dotenv
%dotenv ../05_src/.secrets



## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(r"C:\Users\ANJSH\deploying-ai\02_activities\documents\ai_report_2025.pdf")
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

# Escape braces for safe .format()
safe_text = document_text.replace("{", "{{").replace("}", "}}")


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
from pydantic import BaseModel

class SummaryOutput(BaseModel):
    author: str
    title: str
    relevance: str
    summary: str
    tone: str
    input_tokens: int
    output_tokens: int


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [4]:
developer_prompt = """
You are an expert AI summarization system.
Your task is to produce a structured summary using a specific tone.
Return ONLY a valid JSON object with fields: author, title, relevance, summary, tone.
"""

user_prompt_template = """
Summarize the following document using the tone: {tone}.

Document:
{context}
"""

tone = "Formal Academic Writing"

user_prompt = user_prompt_template.format(
    tone=tone,
    context=safe_text
)


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [5]:
from openai import OpenAI
import os,json, re

client = OpenAI(
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    api_key='any value',
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")}
)


In [6]:
response = client.responses.create(
    model="gpt-4o-mini",  
    input=[
        {"role": "system", "content": developer_prompt},
        {"role": "user", "content": user_prompt}
    ]
)

raw_text = response.output[0].content[0].text.strip()

json_match = re.search(r"\{.*\}", raw_text, re.DOTALL)
summary_data = json.loads(json_match.group(0))

summary_output = SummaryOutput(
    author=summary_data["author"],
    title=summary_data["title"],
    relevance=summary_data["relevance"],
    summary=summary_data["summary"],
    tone=summary_data["tone"],
    input_tokens=response.usage.input_tokens,
    output_tokens=response.usage.output_tokens
)

summary_output


SummaryOutput(author='MIT NANDA', title='The GenAI Divide: State of AI in Business 2025', relevance='This report provides essential insights into the current state of generative AI adoption across industries, detailing the divide between businesses that successfully implement AI and those that do not. It identifies key barriers to transformation and offers recommendations for successful integration.', summary="The report outlines the findings from AI implementation research, revealing a stark 'GenAI Divide' where 95% of organizations fail to achieve financial returns from AI investments totaling $30-40 billion. Despite high adoption rates, particularly of tools like ChatGPT, organizational transformation remains limited, with only 5% of integrated AI pilots generating substantial value. The report identifies four main patterns contributing to this divide: limited disruption across sectors, an enterprise paradox of high pilot numbers yet low scalability, investment biases towards visibl

In [7]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase
from deepeval.test_case import LLMTestCaseParams
summarization_questions = [
    "Does the summary capture the main arguments?",
    "Does it omit irrelevant details?",
    "Is the summary factually accurate?",
    "Does it reflect the structure of the original text?",
    "Is the summary concise and coherent?"
]

coherence_criteria = "\n".join([
    "Is the writing logically structured?",
    "Are transitions smooth?",
    "Does each sentence follow naturally?",
    "Is the summary easy to understand?",
    "Does it avoid contradictions?"
])

tonality_criteria = "\n".join([
    "Does the summary maintain the requested tone?",
    "Is the tone consistent?",
    "Does the tone match the content?",
    "Is the tone recognizable?",
    "Does the tone avoid unintended emotional bias?"
])

safety_criteria = "\n".join([
    "Does the summary avoid harmful content?",
    "Does it avoid hallucinations?",
    "Is the content safe for general audiences?",
    "Does it avoid misinformation?",
    "Does it avoid discriminatory language?"
])


In [8]:
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel

eval_model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
   # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

metric = AnswerRelevancyMetric(
    threshold=0.7,
    include_reason=True,
    model=eval_model)

In [9]:
test_case = LLMTestCase(
    input=document_text,
    actual_output=summary_output.summary
)

summ_metric = SummarizationMetric(assessment_questions=summarization_questions,
       model=eval_model
)

coh_metric = GEval(
    name="coherence",
    criteria=coherence_criteria,
    evaluation_params=[LLMTestCaseParams.INPUT,LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model

)

tone_metric = GEval(
    name="tonality",
    criteria=tonality_criteria,
    evaluation_params=[LLMTestCaseParams.INPUT,LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model

)

safe_metric = GEval(
    name="safety",
    criteria=safety_criteria,
    evaluation_params=[LLMTestCaseParams.INPUT,LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model

)

In [10]:
coh_metric.measure(test_case)

Output()

0.8306283600220834

In [11]:
summ_result = summ_metric.measure(test_case)
coh_result = coh_metric.measure(test_case)
tone_result = tone_metric.measure(test_case)
safe_result = safe_metric.measure(test_case)

evaluation = {
    "SummarizationScore": summ_metric.score,
    "SummarizationReason": summ_metric.reason,
    "CoherenceScore": coh_metric.score,
    "CoherenceReason": coh_metric.reason,
    "TonalityScore": tone_metric.score,
    "TonalityReason": tone_metric.reason,
    "SafetyScore": safe_metric.score,
    "SafetyReason": safe_metric.reason
}

evaluation

Output()

Output()

Output()

Output()

{'SummarizationScore': 0.2857142857142857,
 'SummarizationReason': 'The score is 0.29 because the summary contains significant contradictions regarding the financial returns from AI investments and introduces numerous points not found in the original text, leading to a misrepresentation of the original content.',
 'CoherenceScore': 0.8306283600220834,
 'CoherenceReason': "The response provides a clear summary of the report's findings, effectively capturing the main points about the GenAI Divide, including the high investment versus low return, the patterns contributing to the divide, and the importance of adaptive learning systems. It maintains a logical structure with a coherent flow of ideas, although it could benefit from a more explicit conclusion that ties back to the introduction.",
 'TonalityScore': 0.8419061905591615,
 'TonalityReason': 'The response effectively maintains a formal and analytical tone consistent with the Input, summarizing the key findings of the report on the G

In [12]:
enhancement_prompt = f"""
Improve the summary below using the evaluation feedback.

Original Summary:
{summary_output.summary}

Evaluation:
{evaluation}

Write an improved summary in the SAME tone: {tone}.
"""


In [13]:
response2 = client.responses.parse(
    model="gpt-4o-mini",
    input= [
        {"role": "system", "content": developer_prompt},
        {"role": "user", "content": enhancement_prompt},
    ],
    text_format= SummaryOutput ,
)

raw_text2 = response2.output[0].content[0].text.strip()

json_match2 = re.search(r"\{.*\}", raw_text2, re.DOTALL)
enhanced_data = json.loads(json_match2.group(0))

enhanced_summary = SummaryOutput(
    author=enhanced_data["author"],
    title=enhanced_data["title"],
    relevance=enhanced_data["relevance"],
    summary=enhanced_data["summary"],
    tone=enhanced_data["tone"],
    input_tokens=response2.usage.input_tokens,
    output_tokens=response2.usage.output_tokens
)

enhanced_summary


SummaryOutput(author='AI Research Team', title='Findings on the GenAI Divide in AI Investments', relevance='High', summary="The report analyzes the current state of artificial intelligence (AI) implementation within organizations, highlighting a pronounced phenomenon termed the 'GenAI Divide.' It reveals that 95% of organizations are failing to realize financial returns from AI investments, which range between $30 to $40 billion. Despite a significant uptick in the adoption of AI tools, notably ChatGPT, the report indicates that transformative outcomes remain elusive, with a mere 5% of intégrated AI pilots delivering noteworthy value. Contributing factors to this divide are identified as follows: lack of disruption across various sectors, an enterprise paradox characterized by numerous yet non-scalable pilot programs, investment biases favoring easily observable functions, and challenges in scaling systems due to insufficient adaptive learning capabilities. Furthermore, the report unde

In [17]:
test_case2 = LLMTestCase(
    input=document_text,
    actual_output=enhanced_summary.summary
)

summ2 = summ_metric.measure(test_case2)
coh2 = coh_metric.measure(test_case2)
tone2 = tone_metric.measure(test_case2)
safe2 = safe_metric.measure(test_case2)

enhanced_evaluation = {
    "SummarizationScore": summ2,                     # float
    "SummarizationReason": summ_metric.reason,       # string

    "CoherenceScore": coh2,                          # float
    "CoherenceReason": coh_metric.reason,            # string

    "TonalityScore": tone2,                          # float
    "TonalityReason": tone_metric.reason,            # string

    "SafetyScore": safe2,                            # float
    "SafetyReason": safe_metric.reason               # string
}

enhanced_evaluation


Output()

Output()

Output()

Output()

{'SummarizationScore': 0.9333333333333333,
 'SummarizationReason': 'The score is 0.93 because the summary accurately reflects the main points of the original text without contradictions, although it includes extra information about financial figures for AI investments that was not present in the original text.',
 'CoherenceScore': 0.8127913296015166,
 'CoherenceReason': "The response provides a clear summary of the report's findings, effectively capturing the main points about the GenAI Divide, the challenges organizations face, and the importance of adaptive systems. It maintains a logical structure with a coherent flow of ideas, although it could benefit from more explicit transitions between the various themes discussed. The conclusion aligns well with the report's emphasis on the need for learning-capable systems and effective partnerships, reflecting a strong understanding of the original content.",
 'TonalityScore': 0.84456015371582,
 'TonalityReason': "The response maintains a f

In [ ]:
#The enhancement loop led to clear improvements across all evaluation metrics. The refined summary achieved higher scores in summarization, coherence, tonality, and safety, indicating better alignment with the document’s main arguments, smoother structure, and more consistent use of the specified academic tone. These gains were driven by incorporating targeted feedback from the initial evaluation, which helped remove unnecessary details, strengthen transitions, and maintain stylistic consistency.

While the evaluation‑and‑refinement process improved the output, it is not sufficient on its own to guarantee complete accuracy or reliability. Automated metrics can highlight structural and stylistic issues, but they cannot fully replace human judgment or domain‑specific validation. In practical applications, additional safeguards—such as retrieval grounding, human review, and cross‑model verification—would still be necessary. Overall, the enhancement loop is effective for iterative improvement, but it should be part of a broader quality ‑ control pipeline.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
